In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 5s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [ ]:
!pip install openai

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import numpy as np

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="請填入自己的 API Key"
)

# 對話記憶
chat_history = []

# NPC 知識庫
questions = [
    "How do I beat the China level?",
    "I'm stuck on the China level.",
    "What can I do with the little leaf?",
    "How do I control my character?",
    "How do I beat the Arabia level?",
    "How do I beat the India level?",
    "How do I beat the New Zealand level?",
    "How do I beat the South Africa level?",
    "How do I beat the Antarctica level?",
    "How do I beat the Brazil level?",
    "I keep dying suddenly in the USA level.",
    "How do I beat the USA level?",
    "How do I beat the Italy level?",
    "How do I beat the Russia level?",
    "I'm stuck because I can't jump high enough in the Egypt level.",
    "How do I beat the Egypt level?",
    "How do I beat the Australia level?",
    "I've been stuck on the Australia level for a long time. What should I do?"
]

answers = [
    "To beat the China level, find and touch the scroll. Make sure your health doesn't run out, and don't fall off the map.",
    "Try another route in the China level—you might find the way forward.",
    "The little leaf restores your health.",
    "Use the WASD keys to move and the Spacebar to jump. (Some characters can double jump.)",
    "Stay still and don't press anything.",
    "Be careful of the sudden trash falling from above. Step on the scroll to complete the level.",
    "Be careful of platforms that may collapse beneath you. Find the scroll to complete the level.",
    "You can jump again while airborne to reach higher places.",
    "Leaves restore a lot of HP, so grabbing one is usually worth it even if you're only slightly injured.",
    "Get a feel for the jump height and go for it without hesitation.",
    "No fear, no hesitation—keep going forward!",
    "Take it slow and watch the timing. Move between the falling trash when it's safe.",
    "Don't be scared—I’m here to protect you.",
    "Avoid leaving red tiles on the board. Removing green tiles will recover your health. Use the arrow keys to control your character.",
    "Touch a wall to jump again and climb higher.",
    "Not everything you see is real. Some platforms are only mirages.",
    "Run! Don't get caught by the spreading wildfire.",
    "You'll have a better chance of succeeding if you keep moving while you jump."
]

# NLP 模型
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
question_vectors = model.encode(questions)

def save_memory(player_question, npc_reply):
    chat_history.append(f"玩家：{player_question}")
    chat_history.append(f"NPC：{npc_reply}")

def ask_openrouter(player_question):
    history_text = "\n".join(chat_history[-6:])

    prompt = f"""
You are a cute NPC in an environmental protection game.

Always reply in English.

Reply in a short, cute, game-NPC style.
You can also chat naturally with the player,
but do not make your responses too long.

Previous conversation:
{history_text}

Player says:
{player_question}
"""

    try:
        response = client.chat.completions.create(
            model="openrouter/free",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        npc_reply = response.choices[0].message.content

    except Exception as e:
        print("OpenRouter error：", e)
        npc_reply = "I’m a little tired right now… could you ask me again later？"

    save_memory(player_question, npc_reply)
    return npc_reply

def ask_npc(player_question):
    player_vector = model.encode([player_question])
    scores = cosine_similarity(player_vector, question_vectors)[0]

    best_index = np.argmax(scores)
    best_score = scores[best_index]

    # 相似度高：用自己的知識庫
    if best_score >= 0.45:
        npc_reply = answers[best_index]
        save_memory(player_question, npc_reply)
        return npc_reply

    # 相似度低：交給 AI 閒聊
    return ask_openrouter(player_question)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
while True:
    q = input("玩家：")
    print("NPC：", ask_npc(q))


玩家：中國那關怎過
最接近問題： 中國關卡怎麼過
相似度： 0.8642839
NPC： 中國關卡只要找到卷軸並接觸就能通關。


KeyboardInterrupt: Interrupted by user

In [ ]:
!pip install sentence-transformers scikit-learn flask flask-cors openai

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from threading import Thread

app = Flask(__name__)
CORS(app)

@app.route("/ask", methods=["POST"])
def ask():
    data = request.json
    player_question = data["question"]

    npc_reply = ask_npc(player_question)

    return jsonify({
        "answer": npc_reply
    })

def run_flask():
    app.run(
        host="0.0.0.0",
        port=5005,
        use_reloader=False
    )

Thread(target=run_flask).start()

In [ ]:
!lt --port 5005

your url is: https://eight-emus-stay.loca.lt


INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 02:33:33] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 02:34:25] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 02:56:50] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 02:59:55] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:17:26] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:17:47] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:18:52] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:31:58] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:32:47] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:33:41] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:35:34] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 04:02:01] "POST /ask HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 04:02:28] "POST /ask HT

^C
